In [1]:
import pickle
import re
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import unicodedata
from sentence_transformers import models, SentenceTransformer
     

In [5]:
# Load NLP model
nlp = spacy.load("en_core_web_sm", disable=["ents", "ner", "parser", "pos_"])

In [6]:
# Load Sentence Transormer Model
transformer = models.Transformer("sentence-transformers/all-MiniLM-L12-v2")
pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode="mean")
model = SentenceTransformer(modules=[transformer, pooling])

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [2]:
def clean_preprocess(text):
    with nlp.select_pipes(enable=["tokenizer", "lemmatizer", "attribute_ruler", "tagger"]):
        doc = nlp(text)
        tokens = [
            token.lemma_.lower() for token in doc if
            token.is_ascii and
            not token.is_stop and
            not token.is_punct and
            not token.like_email and
            not token.is_space
        ]
        return " ".join(tokens)

In [12]:
def get_vector(cleaned_text):
    embedding = model.encode(cleaned_text, show_progress_bar=True)

    doc = nlp(str(cleaned_text))
    counts = {"PERSON": 0, "ORG": 0, "GPE": 0, "DATE": 0}
    for ent in doc.ents:
        if ent.label_ in counts:
            counts[ent.label_] += 1
    ner_array = np.array(counts.values())

    return np.concatenate((embedding, ner_array), axis=1)

In [13]:
def apply_model(text, model):
    cleaned_text = clean_preprocess(text)
    vector = get_vector(cleaned_text)
    return model.predict(vector)

In [14]:
# Test Log Reg model
with open("models/RAG-Log-Reg.pkl", "rb") as f:
    loaded_model = pickle.load(f)

    response = ""
    while response != "q":
        response = input("Enter a test prompt: ")

        if apply_model(text=response, model=loaded_model) == 1:
            print("External document retrieval is necessary")
        else:
            print("External document retrieval is not necessary")

/workspaces/cis-405-ragclassifier/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'np' is not defined

In [ ]:
# Test XBoost model
with open("models/RAG-XGBoost.pkl", "rb") as f:
    loaded_model = pickle.load(f)
    
    response = ""
    while response != "q":
        response = input("Enter a test prompt: ")
        embededing = get_embedding(response)
